# Part 4 - Aviation Accident Data Analysis

This notebook reproduces the descriptive analysis, figures, and logistic regression model used in Daniel's Part 4 report section.

In [ ]:
"""
Part 4 - Aviation Accident Data Analysis
ORIE/SYSEN 5200 Team Project
Author: Daniel

Purpose:
    This script reproduces the descriptive analysis, figures, and logistic
    regression model used in Part 4 of the final project report.

Input:
    aviation_accident.csv or aviation_accident(1).csv in /mnt/data

Outputs:
    - Figure 4.1 to Figure 4.5 as PNG files
    - part4_analysis_tables.xlsx
    - regression_summary.csv
    - logistic_regression_coefficients.csv

Notes:
    The model predicts whether an accident is severe. A severe accident is
    defined as an accident with at least one fatal injury OR destroyed aircraft.
    Injury counts and aircraft damage are not used as explanatory variables,
    because they directly define the target variable and would create target leakage.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


# -----------------------------
# 1. File paths
# -----------------------------
BASE_DIR = Path(__file__).resolve().parent
POSSIBLE_INPUTS = [
    BASE_DIR / "aviation_accident.csv",
    BASE_DIR / "aviation_accident(1).csv",
    Path("/mnt/data/aviation_accident.csv"),
    Path("/mnt/data/aviation_accident(1).csv"),
]

DATA_PATH = next((p for p in POSSIBLE_INPUTS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find aviation_accident.csv. Put the CSV in the same folder as this script "
        "or in /mnt/data."
    )

OUT_DIR = BASE_DIR


# -----------------------------
# 2. Helper functions
# -----------------------------
def clean_category(series: pd.Series, missing_label: str = "Unknown/Missing") -> pd.Series:
    """Clean categorical values and replace blank/missing values."""
    cleaned = series.astype("object").where(series.notna(), missing_label)
    cleaned = cleaned.astype(str).str.strip()
    cleaned = cleaned.replace({"": missing_label, "nan": missing_label, "NaN": missing_label})
    return cleaned


def save_bar_chart(data, x_col, y_col, title, xlabel, ylabel, output_name, rotate=0):
    """Save a simple bar chart using matplotlib defaults."""
    plt.figure(figsize=(10, 6))
    plt.bar(data[x_col].astype(str), data[y_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotate, ha="right" if rotate else "center")
    plt.tight_layout()
    plt.savefig(OUT_DIR / output_name, dpi=200)
    plt.close()


def save_line_chart(data, x_col, y_col, title, xlabel, ylabel, output_name):
    """Save a simple line chart using matplotlib defaults."""
    plt.figure(figsize=(10, 6))
    plt.plot(data[x_col], data[y_col], marker="o", linewidth=1)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(OUT_DIR / output_name, dpi=200)
    plt.close()


# -----------------------------
# 3. Load and clean data
# -----------------------------
df = pd.read_csv(DATA_PATH, low_memory=False)

# Parse year. The dataset already has a Year column, but parsing Event.Date is safer.
df["event_date_clean"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df["year_clean"] = df["event_date_clean"].dt.year

# Keep project scope: United States, 1982-2022.
# The source file is already intended to be U.S. accidents, but this keeps the step explicit.
df = df[(df["year_clean"] >= 1982) & (df["year_clean"] <= 2022)].copy()

# Convert injury and numeric fields.
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
]
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")

# Feature engineering: total people and survival rate.
df["total_people"] = df[injury_cols].sum(axis=1)
df["survival_rate"] = np.where(
    df["total_people"] > 0,
    (df["Total.Serious.Injuries"] + df["Total.Minor.Injuries"] + df["Total.Uninjured"])
    / df["total_people"],
    np.nan,
)

# Target: severe accident = fatal injury OR destroyed aircraft.
df["Aircraft.damage.clean"] = clean_category(df["Aircraft.damage"])
df["fatal_any"] = (df["Total.Fatal.Injuries"] > 0).astype(int)
df["destroyed_aircraft"] = (df["Aircraft.damage.clean"].str.lower() == "destroyed").astype(int)
df["severe_accident"] = ((df["fatal_any"] == 1) | (df["destroyed_aircraft"] == 1)).astype(int)

# Clean categorical columns for descriptive analysis and modeling.
df["Weather.Condition.clean"] = clean_category(df["Weather.Condition"])
df["Broad.phase.of.flight.clean"] = clean_category(df["Broad.phase.of.flight"])
df["Purpose.of.flight.clean"] = clean_category(df["Purpose.of.flight"])
df["Engine.Type.clean"] = clean_category(df["Engine.Type"])
df["Amateur.Built.clean"] = clean_category(df["Amateur.Built"])


# -----------------------------
# 4. Descriptive analysis tables
# -----------------------------
accidents_by_year = (
    df.groupby("year_clean")
    .size()
    .reset_index(name="accident_count")
    .sort_values("year_clean")
)

survival_by_year = (
    df.dropna(subset=["survival_rate"])
    .groupby("year_clean")["survival_rate"]
    .mean()
    .reset_index()
    .sort_values("year_clean")
)

phase_summary = (
    df.groupby("Broad.phase.of.flight.clean")
    .agg(
        accidents=("Event.Id", "count"),
        severe_rate=("severe_accident", "mean"),
        fatal_rate=("fatal_any", "mean"),
        avg_survival=("survival_rate", "mean"),
    )
    .reset_index()
    .rename(columns={"Broad.phase.of.flight.clean": "phase"})
    .sort_values("accidents", ascending=False)
)

weather_summary = (
    df.groupby("Weather.Condition.clean")
    .agg(
        accidents=("Event.Id", "count"),
        severe_rate=("severe_accident", "mean"),
        fatal_rate=("fatal_any", "mean"),
        avg_survival=("survival_rate", "mean"),
    )
    .reset_index()
    .rename(columns={"Weather.Condition.clean": "weather"})
    .sort_values("accidents", ascending=False)
)

damage_summary = (
    df.groupby("Aircraft.damage.clean")
    .agg(
        accidents=("Event.Id", "count"),
        severe_rate=("severe_accident", "mean"),
        fatal_rate=("fatal_any", "mean"),
        avg_survival=("survival_rate", "mean"),
    )
    .reset_index()
    .rename(columns={"Aircraft.damage.clean": "damage"})
    .sort_values("accidents", ascending=False)
)

# Save analysis tables.
with pd.ExcelWriter(OUT_DIR / "part4_analysis_tables.xlsx") as writer:
    accidents_by_year.to_excel(writer, sheet_name="accidents_by_year", index=False)
    survival_by_year.to_excel(writer, sheet_name="survival_by_year", index=False)
    phase_summary.to_excel(writer, sheet_name="phase_summary", index=False)
    weather_summary.to_excel(writer, sheet_name="weather_summary", index=False)
    damage_summary.to_excel(writer, sheet_name="damage_summary", index=False)


# -----------------------------
# 5. Figures
# -----------------------------
save_line_chart(
    accidents_by_year,
    "year_clean",
    "accident_count",
    "Number of Aviation Accidents by Year",
    "Year",
    "Accident Count",
    "figure1_accidents_by_year.png",
)

save_line_chart(
    survival_by_year,
    "year_clean",
    "survival_rate",
    "Average Survival Rate by Year",
    "Year",
    "Average Survival Rate",
    "figure2_survival_rate_by_year.png",
)

phase_top = phase_summary.head(12)
save_bar_chart(
    phase_top,
    "phase",
    "accidents",
    "Accident Count by Broad Phase of Flight",
    "Broad Phase of Flight",
    "Accident Count",
    "figure3_accidents_by_phase.png",
    rotate=45,
)

save_bar_chart(
    phase_top,
    "phase",
    "severe_rate",
    "Severe Accident Rate by Broad Phase of Flight",
    "Broad Phase of Flight",
    "Severe Accident Rate",
    "figure4_severity_by_phase.png",
    rotate=45,
)

save_bar_chart(
    weather_summary,
    "weather",
    "severe_rate",
    "Severe Accident Rate by Weather Condition",
    "Weather Condition",
    "Severe Accident Rate",
    "figure5_severity_by_weather.png",
    rotate=30,
)


# -----------------------------
# 6. Logistic regression model
# -----------------------------
# Features are chosen to avoid target leakage.
model_cols = [
    "year_clean",
    "Number.of.Engines",
    "Weather.Condition.clean",
    "Broad.phase.of.flight.clean",
    "Amateur.Built.clean",
    "Purpose.of.flight.clean",
    "Engine.Type.clean",
]

data_model = df[model_cols + ["severe_accident"]].copy()

X = data_model[model_cols]
y = data_model["severe_accident"]

numeric_features = ["year_clean", "Number.of.Engines"]
categorical_features = [
    "Weather.Condition.clean",
    "Broad.phase.of.flight.clean",
    "Amateur.Built.clean",
    "Purpose.of.flight.clean",
    "Engine.Type.clean",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown/Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

logit_model = LogisticRegression(max_iter=1000, class_weight="balanced")

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logit_model),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(y_test, y_proba)
test_accuracy = accuracy_score(y_test, y_pred)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")

regression_summary = pd.DataFrame(
    {
        "n_records_regression": [len(data_model)],
        "severe_rate": [y.mean()],
        "test_auc": [test_auc],
        "test_accuracy": [test_accuracy],
        "cv_auc_mean": [cv_scores.mean()],
        "cv_auc_std": [cv_scores.std()],
        "train_size": [len(X_train)],
        "test_size": [len(X_test)],
    }
)
regression_summary.to_csv(OUT_DIR / "regression_summary.csv", index=False)

# Extract coefficient table.
fitted_preprocessor = clf.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coefs = clf.named_steps["model"].coef_[0]

coef_table = pd.DataFrame(
    {
        "feature": feature_names,
        "coef": coefs,
        "odds_ratio": np.exp(coefs),
        "abs_coef": np.abs(coefs),
    }
).sort_values("abs_coef", ascending=False)

# Clean feature names for readability.
coef_table["feature"] = (
    coef_table["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace(".clean_", "_", regex=False)
)
coef_table.to_csv(OUT_DIR / "logistic_regression_coefficients.csv", index=False)


# -----------------------------
# 7. Console summary
# -----------------------------
print("Part 4 Aviation Accident Data Analysis completed.")
print(f"Input file: {DATA_PATH}")
print(f"Usable records: {len(df):,}")
print(f"Overall average survival rate: {df['survival_rate'].mean():.3f}")
print(f"Fatal accident share: {df['fatal_any'].mean():.3f}")
print(f"Severe accident share: {df['severe_accident'].mean():.3f}")
print(f"Logistic regression test AUC: {test_auc:.3f}")
print(f"5-fold CV AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"Outputs saved to: {OUT_DIR}")
